Reproduces the SAM code (Foret et al. 2021) on Colab.

Before running anything:
Runtime → Change runtime type → T4 GPU.

Run cells in order. Cell 3 restarts the runtime — that is expected;

continue from Cell 4 and do not rerun Cell 3. Setup takes ~10–15 min.

In [1]:
!nvidia-smi

Wed Oct 21 03:27:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%mkdir -p /content/work
%cd /content/work
!git clone -b repro https://github.com/Siddharth-magesh/sam-replication.git sam
!ls sam

/content/work
Cloning into 'sam'...
remote: Enumerating objects: 107, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 107 (delta 1), reused 9 (delta 1), pack-reused 92 (from 1)
Receiving objects: 100% (107/107), 358.76 KiB | 22.42 MiB/s, done.
Resolving deltas: 100% (36/36), done.
autoaugment   docs     __init__.py  OWNERS     repro_notebooks	 sam_jax
CONTRIBUTING  figures  LICENSE	    README.md  requirements.txt


THE RUNTIME WILL RESTART

After the restart, continue from Cell 4. Don’t rerun Cell 3.

In [3]:
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:08
🔁 Restarting kernel...


In [4]:
!mamba create -n sam python=3.8 -y
!mamba install -n sam -c conda-forge cudatoolkit=11.0 "cudnn=8.0.5" -y
!ls /usr/local/envs/sam/lib | grep "libcudnn.so"

[+] 0.0s
[+] 0.0s
[+] 0.1s
conda-forge/linux-64  ⣾  
conda-forge/noarch     1%[+] 0.2s
conda-forge/linux-64   8%
conda-forge/noarch    17%[+] 0.3s
conda-forge/linux-64  12%
conda-forge/noarch    25%[+] 0.4s
conda-forge/linux-64  18%
conda-forge/noarch    38%[+] 0.5s
conda-forge/linux-64  23%
conda-forge/noarch    46%[+] 0.6s
conda-forge/linux-64  29%
conda-forge/noarch    58%[+] 0.7s
conda-forge/linux-64  30%
conda-forge/noarch    65%[+] 0.8s
conda-forge/linux-64  33%
conda-forge/noarch    68%[+] 0.9s
conda-forge/linux-64  36%
conda-forge/noarch    74%[+] 1.0s
conda-forge/linux-64  41%
conda-forge/noarch    83%[+] 1.1s
conda-forge/linux-64  42%
conda-forge/noarch    89%[+] 1.2s
conda-forge/linux-64  45%
conda-forge/noarch    91%[+] 1.3s
conda-forge/linux-64  46%
conda-forge/noarch    95%conda-forge/noarch                                
[+] 1.4s
conda-forge/linux-64  51%[+] 1.5s
conda-forge/linux-64  60%[+] 1.6s
conda-forge/linux-64  65%[+] 1.7s
conda-forge/linux-64  72%[+] 1.8s
conda-

The last line must list libcudnn.so.8. If the install errors, run the original two-step version instead:

In [ ]:
!mamba install -n sam -c conda-forge cudatoolkit=11.0 cudnn -y
!mamba install -n sam -c conda-forge "cudnn=8.0.5" -y

The jaxlib line must read 0.1.57+cuda110, not plain 0.1.57.

In [1]:
PY = "/usr/local/envs/sam/bin/python"

# drop jaxlib, relax the absl-py pin (0.11.0 conflicts with tensorflow-metadata)
!sed -e '/^jaxlib/d' -e 's/^absl-py==0.11.0/absl-py==0.10.0/' /content/work/sam/requirements.txt > /tmp/req_fixed.txt
!grep -E '^(absl|jax)' /tmp/req_fixed.txt

# GPU jaxlib FIRST, from the CUDA-specific index
!{PY} -m pip install "jaxlib==0.1.57+cuda110" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

!{PY} -m pip install -r /tmp/req_fixed.txt
!{PY} -m pip list 2>/dev/null | grep -i -E "^(jax|jaxlib|flax|numpy|tensorflow|absl)"

absl-py==0.10.0
jax==0.2.6
Looking in links: https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.9/157.9 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 147.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 61.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.2/829.2 kB 48.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━

In [2]:
import os, glob
os.environ["LD_LIBRARY_PATH"] = "/usr/local/envs/sam/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

cands = (glob.glob("/usr/local/envs/sam/**/libdevice*.bc", recursive=True)
         or glob.glob("/usr/local/cuda*/nvvm/libdevice/libdevice*.bc"))
print(cands)
os.makedirs("/usr/local/cuda-11.0/nvvm/libdevice", exist_ok=True)
dst = "/usr/local/cuda-11.0/nvvm/libdevice/libdevice.10.bc"
if not os.path.exists(dst):
    os.symlink(cands[0], dst)
print("libdevice link ok:", os.path.exists(dst))

['/usr/local/envs/sam/lib/libdevice.10.bc']
libdevice link ok: True


In [3]:
!/usr/local/envs/sam/bin/python -c "import jax, jax.numpy as jnp; print(jax.devices()); x=jnp.ones((4096,4096)); print((x@x).block_until_ready().sum())" 2>&1 | grep -v -E "dso_loader|cudart|Skipping"
!/usr/local/envs/sam/bin/python -c "import jax; print(jax.random.normal(jax.random.PRNGKey(0), (3,)))" 2>&1 | tail -3
!/usr/local/envs/sam/bin/python -c "import jax.numpy as jnp; from jax import lax; x=jnp.ones((1,8,8,3)); k=jnp.ones((3,3,3,4)); print(lax.conv_general_dilated(x,k,(1,1),'SAME',dimension_numbers=('NHWC','HWIO','NHWC')).shape)" 2>&1 | tail -3

[GpuDevice(id=0)]
68719480000.0
[ 1.8160863  -0.48262316  0.33988908]
(1, 8, 8, 4)


Expected: [GpuDevice(id=0)] and 68719480000.0; three random numbers; (1, 8, 8, 4). Don’t continue unless all three work.

In [4]:
import os, shutil
!mkdir -p /root/tensorflow_datasets
!gdown --folder "https://drive.google.com/drive/folders/1k5Dx5VR6YFmxRf6Id4iNq2gdvYIVbgXj" -O /root/tensorflow_datasets

# if the files landed inside a tfds_cache subfolder, move them up
nested = "/root/tensorflow_datasets/tfds_cache"
if os.path.isdir(nested):
    for n in os.listdir(nested):
        shutil.move(os.path.join(nested, n), "/root/tensorflow_datasets/")

!ls /root/tensorflow_datasets/cifar10/3.0.2

Retrieving folder contents
Retrieving folder 1hJO2P76ykB3u-LUvOptBnm1Vp7PfOL60 cifar10
Retrieving folder 1YQr0vVduxf3m9fhLoL_KYKYrTMwphdi9 3.0.2
Processing file 16A6DAWReYsEG0wl7ndaOYxkdBmTQWT_X cifar10-test.tfrecord-00000-of-00001
Processing file 1BpiBGQl3fEuhQP_rLf3mdcW9Lef1_fWu cifar10-train.tfrecord-00000-of-00001
Processing file 1Z5MCg9SSz4MlfaJfX86SSNNQaw86mamz dataset_info.json
Processing file 1skciIBqF5hg0q2Kgm5iGnSyeAxg8Lh8t features.json
Processing file 1xW1Qp9XZjTnqHNrXMuBEI5ZSaYg8JZWL label.labels.txt
Retrieving folder 1W_BhXvnTTsrQ9wlMqvYKUPu3M5IiE13h downloads
Retrieving folder 1D4sUCPnUkAPbev-3RKvLt4ZwkuTGbVT8 extracted
Retrieving folder 1PY0yEdnZjVfF-6ndRSQRo3LzDx3PSCjV TAR_GZ.cs.toronto.edu_kriz_cifar-10-binaryODHPtIjLh3oLcXirEISTO7dkzyKjRCuol6lV8Wc6C7s.tar.gz
Retrieving folder 1wBR51lPPD38EjFA5gRtsAtFCK0vnpLP- cifar-10-batches-bin
Processing file 1sqD22r3kD5w2NfiPhPDhXeY5IvIXbV1f batches.meta.txt
Processing file 19G0Usv9IoPOjkHGFpOlGzHpFdKlQAcD9 data_batch_1.bin
Proce

The last line should list train/test tfrecord files and dataset_info.json. If the download fails (quota error or folder not found), the training step will download CIFAR-10 itself, which takes ~38 minutes.

In [5]:
%cd /content/work
!mkdir -p /content/runs
!/usr/local/envs/sam/bin/python -m sam.sam_jax.train \
  --dataset cifar10 --model_name WideResnet28x10 \
  --output_dir /content/runs/smoke_sam \
  --num_epochs 1 --batch_size 256 \
  --learning_rate 0.1 --weight_decay 0.0005 \
  --sam_rho 0.05 --run_seed 0 > /content/runs/smoke_sam.log 2>&1
!tail -5 /content/runs/smoke_sam.log
!grep -h "finished in" /content/runs/smoke_sam.log

/content/work
I1021 03:42:36.625574 133953210431296 flax_training.py:858] Epoch 0 finished in 532.52s.
I1021 03:42:36.625689 133953210431296 flax_training.py:863] Evaluating at end of epoch 0 (0-indexed)
I1021 03:42:57.444298 133953210431296 flax_training.py:897] Evaluated model in 20.82.
I1021 03:42:57.916910 133953210431296 checkpoints.py:90] Saving checkpoint at step: 0
I1021 03:42:59.694583 133953210431296 checkpoints.py:99] Saved checkpoint at /content/runs/smoke_sam/lr_0.1/wd_0.0005/rho_0.05/seed_0/checkpoints/checkpoint_0
I1021 03:42:36.625574 133953210431296 flax_training.py:858] Epoch 0 finished in 532.52s.


The cell shows nothing until it finishes. Don’t interrupt it.

In [6]:
!/usr/local/envs/sam/bin/python -m sam.sam_jax.train \
  --dataset cifar10 --model_name WideResnet28x10 \
  --output_dir /content/runs/smoke_sgd \
  --num_epochs 1 --batch_size 256 \
  --learning_rate 0.1 --weight_decay 0.0005 \
  --sam_rho 0.0 --run_seed 0 > /content/runs/smoke_sgd.log 2>&1
!grep -h "finished in" /content/runs/smoke_sgd.log

I1021 03:48:02.211035 137060814817088 flax_training.py:858] Epoch 0 finished in 282.29s.


--sam_rho 0.0 means plain SGD.

In [7]:
%%writefile /content/read_tb.py
import glob
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
from tensorboard.util import tensor_util
for d in sorted(glob.glob("/content/runs/*/lr_*/wd_*/rho_*/seed_*")):
    ea = EventAccumulator(d, size_guidance={"tensors": 0}); ea.Reload()
    print("==", d)
    for t in ["train_error_rate", "test_error_rate", "test_loss"]:
        ev = ea.Tensors(t)
        print("  ", t, [(e.step, round(float(tensor_util.make_ndarray(e.tensor_proto)), 4)) for e in ev])

Writing /content/read_tb.py


In [8]:
!/usr/local/envs/sam/bin/python /content/read_tb.py 2>&1 | grep -v -E "dso_loader|cudart|Skipping"

== /content/runs/smoke_sam/lr_0.1/wd_0.0005/rho_0.05/seed_0
   train_error_rate [(195, 0.5814)]
   test_error_rate [(195, 0.4281)]
   test_loss [(195, 1.192)]
== /content/runs/smoke_sgd/lr_0.1/wd_0.0005/rho_0.0/seed_0
   train_error_rate [(195, 0.5581)]
   test_error_rate [(195, 0.42)]
   test_loss [(195, 1.1716)]


Save results to your Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/sam_runs
!cp -r /content/runs/* /content/drive/MyDrive/sam_runs/

Open the notebook, Runtime → Change runtime type → T4 GPU, then Save.

Run the cells in order. Cell 3 restarts the runtime. That is normal. Continue from Cell 4 and don’t rerun Cell 3.

Stop and message me if Cell 7 doesn’t show [GpuDevice(id=0)], 68719480000.0 and (1, 8, 8, 4).

Cells 9 and 10 look idle for several minutes. Don’t interrupt them.

Success: both runs print Epoch 0 finished in ... s, and Cell 11 shows a test_error_rate well below 0.9 (ours was roughly 0.35–0.45 after 1–2 epochs; this only checks the pipeline). Send me your two epoch times and the GPU type.
If anything fails, send the last 30 lines of the error, not the whole log.